In [ ]:
##combining all together in one file (final api) for simplicity, but in production these would be separate modules

from fastapi import FastAPI
from datetime import datetime, timezone, timedelta

from grid_intelligence.services.entsoe_client import fetch_day_ahead_prices, parse_prices
from grid_intelligence.logic.model import model
from grid_intelligence.logic.preprocessor import create_features

app = FastAPI()

@app.get("/predict-live")
def predict_live():

    # 1. Get ENTSO-E data
    end = datetime.now(timezone.utc)
    start = end - timedelta(days=2)

    start_str = start.strftime("%Y%m%d%H%M")
    end_str = end.strftime("%Y%m%d%H%M")

    xml = fetch_day_ahead_prices(start_str, end_str)
    df = parse_prices(xml)

    # 2. Feature engineering
    df_feat, features = create_features(df)

    # 3. Get latest row
    X = df_feat[features].iloc[-1:].values

    # 4. Predict
    prediction = model.predict(X)[0]

    return {
        "prediction": float(prediction)
    }


In [ ]:
## for preprocessor.py file

def create_features(df):

    df = df.copy()

    # Example features (adapt to your dataset)
    df["hour"] = df["ds"].dt.hour
    df["day_of_week"] = df["ds"].dt.dayofweek

    # Lags (important for energy prices)
    df["lag_1"] = df["y"].shift(1)
    df["lag_24"] = df["y"].shift(24)

    df = df.dropna()

    feature_cols = ["hour", "day_of_week", "lag_1", "lag_24"]

    return df, feature_cols

In [ ]:
#### for model.py file

import joblib
import os

MODEL_PATH = os.path.join(
    os.path.dirname(__file__),
    "model.pkl"
)

model = joblib.load(MODEL_PATH)